# Project 1 — Teach a Neural Network to Predict Crime Rates

### AI Builders Lab · Class 2 · August 23

Last week you saw AI models being *used*. Today you **build** one.

That difference is the whole point of this class. Downloading somebody's finished model and
running it is like driving a car. Today we are going to assemble the engine.

By the end of this notebook you will have:

1. Loaded a real dataset of **600 cities** and looked at it honestly
2. Found and removed **bad data** that would poison the model
3. Built a neural network layer by layer
4. **Trained it** — watched the error fall, live, on your screen
5. Fed it a city it has never seen and asked it to predict the crime rate

Nothing here is pre-trained. Every number in this model will be learned on your machine,
in front of you, in about twenty seconds.

---
### How to use this notebook
Click on a cell and press **Shift + Enter** to run it. Run them **in order, top to bottom**.
If you skip one, later cells will fail — Python has no memory of code you did not run.

---
# Part 0 — What is Google Colab, and why are we in it?

**Colab is a Python notebook that runs on Google's computers, not yours.**

You open it in a browser. Google gives you a temporary machine with Python, TensorFlow,
and a graphics card already installed. Nothing to install, nothing to break, and it works
identically on a Chromebook and a gaming PC.

Two reasons we teach in it:

1. **The major AI competitions require it.** USA AIO (the USA AI Olympiad) expects every
   answer and every program to be written and run in Colab. Learning the tool *is* part of
   the preparation.
2. **Everyone gets the same environment.** No "it works on my laptop" problems.

### Choosing your hardware — do this now

Go to **Runtime → Change runtime type**. You will see a choice of *CPU*, *T4 GPU*, and others.

| | What it is | When you need it |
|---|---|---|
| **CPU** | The ordinary processor. A few strong workers. | Small data, small models. **Everything today.** |
| **GPU** | A graphics card. Thousands of weak workers doing the same sum at once. | Images, big networks, anything that takes more than a few minutes on CPU |

A neural network's core operation is multiplying big grids of numbers, over and over.
That is exactly the job a GPU was built for. But GPUs are a shared, limited resource in
free Colab — **do not ask for one you do not need.** For today, **CPU is correct and faster
to start.**

### One more thing, and it is the most important sentence in this class

**An AI must be trained before it can be used.** Training is where the model looks at
examples and adjusts its own internal numbers until its answers stop being wrong. Using is
what happens after. Last week we used. Today we train.

---
# Part 1 — Open the toolbox

Python on its own does not know what a neural network is. We have to `import` the tools.

Think of it like laying out equipment on a lab bench before an experiment:

| Tool | What it does for us |
|---|---|
| `pandas` | Reads spreadsheets and tables. Our data is a table, so this is how we open it. |
| `numpy` | Handles arrays of numbers. Every value inside a network is one of these. |
| `matplotlib` | Draws graphs. We use it to watch the model learn. |
| `tensorflow` / `keras` | Builds and trains the neural network. This is the engine. |
| `scikit-learn` | A box of classic tools. We use exactly one: the train/test splitter. |

**Type this cell yourself.** It is the only one I will ask you to type — everything after it
you can paste. Typing it once tells your fingers that this is just Python.

In [ ]:
# Run this cell first  (Shift + Enter)

import pandas as pd                                    # tables and spreadsheets
import numpy as np                                     # arrays of numbers
import matplotlib.pyplot as plt                        # graphs

import tensorflow as tf
from tensorflow.keras.models import Sequential         # "layers stacked in a row"
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam           # Adam = "adaptive moment estimation"
from sklearn.model_selection import train_test_split

# Fix the random seed. Neural networks start from random numbers, so without this
# your result and your neighbour's would differ every single run.
tf.keras.utils.set_random_seed(42)

print("TensorFlow version:", tf.__version__)
print("Toolbox is open.")

---
# Part 2 — Get the data, then *look* at it

We are using real US city crime statistics: **600 cities**, each described by five numbers.

The question we are asking the model is:

> Given how much a city spends on police and how educated its population is,
> how much crime gets reported there?

Notice this is a **regression** problem — the answer is a *quantity* (crimes per million
residents), not a category. Project 2 later today will be the other kind: a *classification*
problem, where the answer is one of ten labels.

In [ ]:
# The data lives in our class GitHub repository, so pandas can read it straight off the web.
# No downloading, no Google Drive, no uploading. One line.

DATA_URL = "https://raw.githubusercontent.com/chizelnut/ABL_Aug_23/main/data/crimeSTATS.csv"

crime_data = pd.read_csv(DATA_URL, sep=',')

display(crime_data)          # in Colab, display() renders a table nicely


### Stop. Read the table above before you do anything else.

This is the single most skipped step in machine learning and it is where half of all bugs live.

**Look at the shape of it:**

- Each **column** going down is a **feature** — one measurable property, one *kind* of fact
  about a city. `annual_police_funding_per_res` is a feature. `%_college_degree` is a feature.
  Five features here.
- Each **row** going across is a **sample** — one real city, one complete example that
  actually happened. 600 samples here.

That is the whole mental model of tabular data. **Columns are what you measure.
Rows are who you measured.** Every dataset you meet for the rest of your life is some
version of this, and every AI framework expects data in exactly this shape.

One of those columns is special. `total_crime_reported_per_1_million_res` is the thing we want
to *predict* — the **target**, or **label**, or **Y**. The other five are the **inputs**, or **X**.
The model's whole job is to learn the route from X to Y.

In [ ]:
# How big is this thing, and what is in it?

print("Shape (rows, columns):", crime_data.shape)
print("  ->", crime_data.shape[0], "SAMPLES (cities), described by", crime_data.shape[1], "columns")

target = 'total_crime_reported_per_1_million_res'

print("\nThe TARGET - the thing we want to predict (Y):")
print("   ", target)

print("\nThe FEATURES - the facts we get to look at (X):")
for i, name in enumerate(crime_data.columns.drop(target)):
    print("   ", i, name)

print("\nThe first 3 rows - these are SAMPLES, three real cities:")
display(crime_data.head(3))

---
# Part 3 — Clean the data

Raw data is never ready. Before we train anything we ask two questions:

1. **Is anything missing?** A blank cell will crash training or silently poison it.
2. **Is anything impossible?** Values that could not exist in the real world.

Question 2 is the one people forget, and it is the one that bites. A model has no common
sense. It cannot tell that a number is absurd — it will dutifully learn from it.

In [ ]:
# Question 1: any missing values?
print("Missing values per column:")
print(crime_data.isnull().sum())

In [ ]:
# Question 2: any impossible values?
# A city cannot have a NEGATIVE number of crimes. Let's look.

print("Lowest crime value in the data:", crime_data[target].min())
print("\nRows where crime is negative — these cannot be real:")
display(crime_data[crime_data[target] < 0])

**Four cities report negative crime.** That is not a small crime rate; it is a
data-entry error, a broken sensor, or a placeholder someone forgot to remove.

If we leave them in, the model spends part of its capacity learning to predict impossible
numbers, and it will drag every nearby prediction downward.

We remove them. Four rows out of 600 — we keep 99.3% of our data and lose all of the poison.

In [ ]:
rows_before = len(crime_data)

crime_data = crime_data[crime_data[target] >= 0]

print("Rows before cleaning:", rows_before)
print("Rows after cleaning: ", len(crime_data))
print("Removed:", rows_before - len(crime_data), "impossible rows")

---
# Part 4 — Split into X and Y, then into training and testing

### First: separate the question from the answer

`X` = the five features (what the model gets to see).
`Y` = the crime rate (what it has to figure out).

In [ ]:
X = crime_data.drop([target], axis=1).values     # everything EXCEPT the target -> the inputs
Y = crime_data[[target]].values                  # ONLY the target -> the answer

print("X shape:", X.shape, " <- 596 cities, 5 features each")
print("Y shape:", Y.shape, " <- 596 answers")

print("\nCity number 0 looks like this to the model:")
print("  X =", X[0], "  -> Y =", Y[0][0])

### Second: hold some data back

We split into **80% for training** and **20% for testing**, and the model never sees the
test 20% while it learns.

**Why bother? Why not train on everything?**

Because a neural network with enough capacity can simply *memorise*. If we tested it on
data it had already studied, it could score beautifully by recall alone and we would learn
nothing about whether it can handle a new city.

It is the same reason your teacher does not hand out the exam paper as the homework.
The point of a test is to measure what you can do with problems you have not seen.

`random_state=25` fixes which rows land in which pile, so your split matches your
classmates' and your results are comparable.

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.20,        # 20% held back for the exam
    random_state=25        # a fixed random seed, so the split is reproducible
)

print("Training on:", X_train.shape[0], "cities")
print("Testing on: ", X_test.shape[0], "cities  (the model will not see these until the end)")

---
# Part 5 — Build the network

Now we design the brain. Look at the diagram on the screen while you read this — the code
below is that picture, written down.

Our architecture, and the reason for each piece:

| Layer | Size | Why this |
|---|---|---|
| `Input` | 5 | One slot per feature. The model has to be told the shape of one sample. |
| `Dense` + ReLU | 100 | The first and widest hidden layer. "Dense" means every one of the 5 inputs connects to every one of the 100 neurons. |
| `Dropout` | 0.1 | While training, randomly switch off 10% of the connections each step. |
| `Dense` + ReLU | 50 | Narrowing. Each layer compresses what it received into fewer, more useful summaries. |
| `Dropout` | 0.1 | |
| `Dense` + ReLU | 25 | Narrowing further. |
| `Dropout` | 0.1 | |
| `Dense` | **1** | One neuron, **no activation function**. This is the predicted crime rate. |

### Three choices worth understanding

**Why ReLU?** ReLU is `max(0, x)` — it passes positive numbers through untouched and
flattens everything negative to zero. We want it here because crime counts have **no upper
bound**. A sigmoid would squash every value into 0–1 and cap our predictions. ReLU lets the
number grow as large as the data demands.

**Why Dropout?** During training it randomly sets some connection weights to zero, so the
network cannot lean on any single neuron. It is forced to spread the knowledge around.
This is our defence against **overfitting** — memorising the training cities instead of
learning the pattern. Note it only happens during training; at prediction time all neurons work.

**Why does the last layer have no activation?** Because we want a raw number out.
Any activation would constrain the range. This is the signature of a regression network —
one output neuron, no activation. In Project 2 you will see the classification version and
it looks completely different.

In [ ]:
def create_model(learning_rate, dropout_rate):

    model = Sequential()                       # Sequential = layers in a straight line, one after another

    model.add(Input(shape=(X_train.shape[1],)))       # 5 features in

    model.add(Dense(100, activation='relu'))          # why ReLU? no upper bound for crime incidents
    model.add(Dropout(dropout_rate))                  # randomly zero some weights, to avoid overfitting

    model.add(Dense(50,  activation='relu'))
    model.add(Dropout(dropout_rate))

    model.add(Dense(25,  activation='relu'))
    model.add(Dropout(dropout_rate))

    model.add(Dense(1))                               # one number out. No activation - we want it unbounded.

    adam = Adam(learning_rate=learning_rate)
    model.compile(loss='mean_squared_error',          # MSE: the standard cost function for regression
                  optimizer=adam,
                  metrics=['mae'])                    # MAE: "on average, how far off are we?" - in real units
    return model


# The settings WE choose. These are called hyper-parameters:
# the model does not learn them, a human picks them.
dropout_rate  = 0.1
learn_rate    = 0.01
epochs        = 60

model = create_model(learn_rate, dropout_rate)
model.summary()

### Read that summary — the parameter count is the point

`Total params` is the number of individual numbers this network will adjust while learning.
Nobody sets any of them by hand.

Check the first layer: **600 parameters**. Where does that come from?

$$5 \text{ inputs} \times 100 \text{ neurons} = 500 \text{ weights}, \quad +\ 100 \text{ biases} = 600$$

The rule for any Dense layer is:

$$\text{parameters} = (\text{neurons in} \times \text{neurons out}) + \text{neurons out}$$

A **weight** decides how much each incoming signal matters — like voting power.
A **bias** shifts the neuron, so it can still fire even when every input is zero.
Those two kinds of numbers are the *only* things a network learns. Training is nothing but
repeatedly adjusting weights and biases.

Hold onto this formula. It is how you will one day make sense of "GPT has 175 billion parameters."
Same arithmetic, more layers.

---
# Part 6 — Train it

This is the moment. Everything before now was preparation.

- **`epochs=60`** — go through all the training cities 60 times. One full pass is one epoch.
  The network improves a little on each pass, like re-reading a chapter.
- **`batch_size=16`** — look at 16 cities, then update the weights. Then the next 16.
  Updating after every single city would be very slow; updating only once per pass would be
  too crude. 16 is a normal compromise.
- **`validation_split=0.2`** — carve another 20% out of the *training* data as a progress check
  during training. This is separate from `X_test`, which stays sealed.
- **`verbose=1`** — show me the progress bar. Watch it.

**Watch the `loss` and `mae` columns as it runs.** You are literally watching the model get
less wrong. That falling number is learning, happening.

In [ ]:
model_history = model.fit(
    X_train, Y_train,
    batch_size=16,
    epochs=epochs,
    validation_split=0.2,
    verbose=1
)

print("\nTraining finished. The model now holds", model.count_params(), "learned numbers.")

---
# Part 7 — How well did it do?

Now we open the sealed envelope: the 20% of cities the model has never seen.

**MAE (Mean Absolute Error)** is the number to read. It answers "on average, how far off is
the prediction?" — in real units, crimes per million residents.

In [ ]:
score = model.evaluate(X_test, Y_test, verbose=0)

print("Loss (MSE):          ", round(score[0], 1))
print("Mean Absolute Error: ", round(score[1], 1), "crimes per million residents")

# Is that good? Compare against the laziest possible model: always guess the average.
naive = float(np.abs(Y_test - Y_train.mean()).mean())
print("\nA model that just guesses the average every time would be off by:", round(naive, 1))
print("So our network is roughly", round(naive / score[1], 1), "times better than guessing.")

**Always compare against a stupid baseline.** An accuracy number on its own means nothing.
"Off by 85" sounds bad until you know that guessing the average is off by 182.

This habit will save you in competitions: before you celebrate a score, ask what the trivial
solution scores.

In [ ]:
# Graph the error for the training set and the validation set

plt.figure(figsize=(9, 5))
plt.plot(model_history.history['mae'], label='training data')
plt.plot(model_history.history['val_mae'], label='validation data (held out)')
plt.legend(loc='upper right')
plt.title('Model Error over Training')
plt.ylabel('Mean Absolute Error')
plt.xlabel('Epoch')
plt.grid(alpha=0.3)
plt.show()

### Reading this graph

Both lines should fall steeply and then flatten. That flattening is the model running out
of things to learn from this data.

Look at the **gap between the two lines**:

- **Close together** → the model is learning a real pattern. Good.
- **Training line far below the validation line** → **overfitting**. It is memorising the
  training cities instead of learning what drives crime.
- **Validation line turning back upward** → you have trained too long. Stop earlier.

You will read this same graph for every model you ever train.

---
# Part 8 — Predict a city that does not exist

The model is trained. Now we use it — and notice how short this part is. Building and
training was the work; using is one line.

Let us invent a city:

| Feature | Value |
|---|---|
| annual police funding per resident | 55 |
| % of over-25s who finished high school | 68 |
| % of 16–19 year-olds not in high school | 14 |
| % of 18–24 year-olds in college | 26 |
| % with a college degree | 17 |

**The order of these five numbers must match the column order exactly.** The model has no
idea what the columns are called — it only knows position 0, position 1, and so on. Swap two
and it will give you a confident, wrong answer with no error message. This is the most common
beginner bug in the entire field.

In [ ]:
# A 2-D array, because the model expects a BATCH of samples - even a batch of one.
X_new = np.array([[55, 68, 14, 26, 17]], dtype=X_train.dtype)

prediction = model.predict(X_new, verbose=0)

print("Feature order the model expects:")
print("  ", list(crime_data.drop([target], axis=1).columns))
print("\nOur made-up city:", X_new[0])
print("\nPREDICTED total crime per 1 million residents:", round(float(prediction[0, 0]), 1))

In [ ]:
# Now change something and see what the model believes.
# Here: the same city, but with police funding raised from 55 to 80.

X_new_2 = np.array([[80, 68, 14, 26, 17]], dtype=X_train.dtype)
print("With police funding at 55:", round(float(model.predict(X_new,   verbose=0)[0,0]), 1))
print("With police funding at 80:", round(float(model.predict(X_new_2, verbose=0)[0,0]), 1))

**Careful with what you just did.** The model learned that in this dataset, higher police
funding goes together with a higher crime level. That is a **correlation**, not a cause.
Cities with more crime tend to spend more on police *because* they have more crime.

A neural network cannot tell the difference between "A causes B" and "A and B happen together."
It only ever learns patterns of co-occurrence. Keep that firmly in mind before anyone tells
you an AI proved something.

---
# Part 9 — Save the model

Training took twenty seconds here. Real models take days or weeks on expensive hardware.
You never want to retrain something you already have.

`model.save()` writes every learned weight and bias to a single file.

In [ ]:
model.save('crime_model.keras')
print("Saved to crime_model.keras")

# Click the FOLDER icon in the left sidebar of Colab to see it.
# Colab deletes this storage when your session ends, so right-click -> Download to keep it.

# To load it back later, in any notebook:
# from tensorflow import keras
# model = keras.models.load_model('crime_model.keras')

---
# What you just did

1. Opened a real dataset and **looked at it** — columns are features, rows are samples
2. Found four impossible rows and **removed them**
3. Split 80/20 so the test would be honest
4. Built a 5 → 100 → 50 → 25 → 1 network and counted its parameters
5. **Trained it** — you watched the error fall, live
6. Checked it against a stupid baseline, then predicted a city that does not exist

That is the complete machine learning workflow. Every project in this course, and every
problem in the AI Olympiad, is this same seven-step shape with different data.

**Now open Project 2.** Same skeleton, completely different problem: instead of predicting
a *number*, the model will have to pick one of *ten categories* — and the input will be a
picture.